### New attention mechanism

Given, dot-product selection attention: takes cosine similarity between data vectors.

Output: row-wise stochastic score matrix $Softmax_{row}(S)$

We need a replacement for always finding cosine overlap between Q-K and further S-V.

How about letting a learned func ($f_a$) figure out mapping between elements. So the embeddings dont have to closer or its spatial position is not the exlusive factor for attention.

The $f_a$ carries inductive bias for a domain. For example, in robot trajectory (obs.state+obs.delta_goal to action.delta_ee_pos) in vpg rl setting.



The inductive bias in robot trajectory is it, motion is continous and has to be physically feasible.

Learn method first, then use method for attending obs

In [2]:
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim

In [ ]:
class ScorerModel(nn.Module):
    """mlp_k + predict + learnable temperature τ."""

    def __init__(self, obs_dim: int, d_z: int):
        super().__init__()
        self.mlp_k = nn.Sequential(
            nn.Linear(obs_dim, d_z * 2),
            nn.ReLU(),
            nn.Linear(d_z * 2, d_z),
        )
        self.predict = nn.Sequential(
            nn.Linear(d_z, d_z * 2),
            nn.ReLU(),
            nn.Linear(d_z * 2, d_z),
        )
        self.tau = nn.Parameter(torch.ones(1))

    def loss(self, obs: torch.Tensor) -> torch.Tensor:
        """
        obs: (B, T, obs_dim)
        Pairs: (obs_t, obs_{t+1}) for t in 0..T-2.
        L = mean ‖predict(mlp_k(obs_t)) − sg(mlp_k(obs_{t+1}))‖²
        """
        obs_t  = obs[:, :-1].reshape(-1, obs.shape[-1])   # (B*(T-1), obs_dim)
        obs_tp1 = obs[:, 1:].reshape(-1, obs.shape[-1])

        z_t   = self.mlp_k(obs_t)                         # (B*(T-1), d_z)
        z_tp1 = self.mlp_k(obs_tp1).detach()              # stop-gradient on target

        pred  = self.predict(z_t)                          # (B*(T-1), d_z)
        return F.mse_loss(pred, z_tp1)
    
    def freeze(self):
        for p in self.parameters():
            p.requires_grad_(False)

    def score(
        self,
        o_q: torch.Tensor,  # (B, n_heads, T_q, obs_dim) — query pseudo-obs
        o_k: torch.Tensor,  # (B, n_heads, T_k, obs_dim) — key pseudo-obs
    ) -> torch.Tensor:      # (B, n_heads, T_q, T_k)
        B, H, T_q, obs_dim = o_q.shape
        _, _, T_k, _ = o_k.shape

        # Flatten batch and head dims for the shared MLP
        o_q_flat = o_q.reshape(B * H * T_q, obs_dim)
        o_k_flat = o_k.reshape(B * H * T_k, obs_dim)

        z_q = self.mlp_k(o_q_flat).reshape(B, H, T_q, -1)  # (B, H, T_q, d_z)
        z_k = self.mlp_k(o_k_flat).reshape(B, H, T_k, -1)  # (B, H, T_k, d_z)

        # Forward roll on keys only
        d_z = z_k.size(-1)
        z_k_rolled = self.predict(z_k.reshape(B * H * T_k, d_z))
        z_k_rolled = z_k_rolled.reshape(B, H, T_k, d_z)      # (B, H, T_k, d_z)

        # L2 distance: expand to (B, H, T_q, T_k)
        # z_q: (B, H, T_q, 1, d_z), z_k_rolled: (B, H, 1, T_k, d_z)
        diff = z_q.unsqueeze(3) - z_k_rolled.unsqueeze(2)    # (B, H, T_q, T_k, d_z)
        dist_sq = (diff ** 2).sum(dim=-1)                     # (B, H, T_q, T_k)

        return -dist_sq / self.tau.abs().clamp(min=1e-6)


In [4]:
class Transformer(nn.Module):
    def __init__(self, cfg: CFG, scorer: Optional[nn.Module] = None):
        super().__init__()
        self.input_proj = nn.Linear(cfg.obs_dim, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.seq_len, cfg.d_model)
        self.blocks = nn.ModuleList([
            FDCATransformerBlock(
                d_model=cfg.d_model,
                n_heads=cfg.n_heads,
                obs_dim=cfg.obs_dim,
                ffn_mult=cfg.ffn_mult,
                scorer=scorer,
                causal=cfg.causal,
                dropout=cfg.dropout,
            )
            for _ in range(cfg.n_layers)
        ])
        self.ln_out = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.obs_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, obs_dim)  →  (B, T, obs_dim)"""
        B, T, _ = x.shape
        positions = torch.arange(T, device=x.device)
        h = self.input_proj(x) + self.pos_emb(positions)
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln_out(h))


NameError: name 'CFG' is not defined